# CSoT'26 — ML in Astronomy — Week 1 · Part 2: Data Pipeline (Solution)

Reference implementation. **Only open after attempting [`week1_data_starter.ipynb`](week1_data_starter.ipynb).**

Companion reading: [`08-data-pipelines.md`](../08-data-pipelines.md).

## Step 0 — Imports

In [ ]:
import os
import random
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import torchvision
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

## Step 1 — Get the dataset into Colab

**What you download**

| File | Where it comes from | What it contains |
|------|---------------------|------------------|
| `images_gz2.zip` | Kaggle bundle | Flat JPGs: `1.jpg`, `2.jpg`, … |
| `gz2_filename_mapping.csv` | Kaggle bundle | `objid` ↔ `asset_id` |
| `gz2_hart16.csv` | [data.galaxyzoo.org](https://data.galaxyzoo.org/) | `dr7objid` + `gz2_class` (e.g. `Sc2t`, `Ei`) |

There are **no class subfolders** in the raw download. We build them in Step 3.

In [ ]:
RAW_ROOT = Path("galaxy_raw")
IMAGES_DIR = RAW_ROOT / "images_gz2"   # adjust if your JPGs landed one folder deeper
DATA_ROOT = Path("galaxy_data")        # we create train/val/test subfolders here
LABELS_URL = "https://gz2hart.s3.amazonaws.com/gz2_hart16.csv.gz"

# --- Download via Kaggle API (run once) ---
# from google.colab import files
# files.upload()  # select kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip -q install kaggle pandas
# !kaggle datasets download -d jaimetrickz/galaxy-zoo-2-images
# !unzip -q -o galaxy-zoo-2-images.zip -d {RAW_ROOT}
# !unzip -q -o {RAW_ROOT}/images_gz2.zip -d {IMAGES_DIR}
# !wget -q -O {RAW_ROOT}/gz2_hart16.csv.gz {LABELS_URL}
# !gunzip -f {RAW_ROOT}/gz2_hart16.csv.gz

print("RAW_ROOT   =", RAW_ROOT)
print("IMAGES_DIR =", IMAGES_DIR)
print("DATA_ROOT  =", DATA_ROOT)

## Step 2 — Inspect the raw layout

You should see a **flat** image folder plus CSVs — not `elliptical/`, `spiral/`, etc.

If `IMAGES_DIR` is empty, check whether JPGs are nested one level deeper (e.g. `images_gz2/images_gz2/`) and update `IMAGES_DIR`.

In [ ]:
print("RAW_ROOT contents:", sorted(p.name for p in RAW_ROOT.iterdir()))
jpg_count = sum(1 for _ in IMAGES_DIR.glob("*.jpg"))
print(f"Flat JPG count in {IMAGES_DIR}: {jpg_count:,}")

print("\nMapping CSV preview:")
print(pd.read_csv(RAW_ROOT / "gz2_filename_mapping.csv", nrows=3))

print("\nLabels CSV preview (note dr7objid — we rename to objid before merging):")
print(pd.read_csv(RAW_ROOT / "gz2_hart16.csv", nrows=3)[["dr7objid", "gz2_class"]])

## Step 3 — Join the CSVs (labels still aren't folders yet)

Think of it as a three-table lookup:

```
asset_id  ──mapping CSV──►  objid  ──hart16 CSV──►  gz2_class  ──collapse──►  label
   16          5877…           5877…                  Sc2t                    spiral
```

Run the cell below and check:
- `len(df)` is in the tens/hundreds of thousands (not zero)
- `label` counts show elliptical / spiral / spiral_barred

In [ ]:
def high_level_label(gz2_class: str):
    """Collapse detailed GZ2 codes (Sc2t, Ei, SBb2m, …) to a few training buckets."""
    if not isinstance(gz2_class, str) or gz2_class == "A":
        return None  # artifact / ambiguous
    if gz2_class.startswith("E"):
        return "elliptical"
    if gz2_class.startswith("SB"):
        return "spiral_barred"
    if gz2_class.startswith("S"):
        return "spiral"
    return None


def load_labeled_table(mapping_csv, labels_csv):
    """Join Kaggle mapping (objid ↔ asset_id) with Hart et al. morphology labels."""
    mapping = pd.read_csv(mapping_csv)
    labels = pd.read_csv(labels_csv)
    if "dr7objid" in labels.columns:
        labels = labels.rename(columns={"dr7objid": "objid"})
    df = mapping.merge(labels[["objid", "gz2_class"]], on="objid", how="inner")
    df["label"] = df["gz2_class"].map(high_level_label)
    df = df.dropna(subset=["label"]).reset_index(drop=True)
    return df


def _link_image(src: Path, dst: Path) -> bool:
    """Symlink if possible; otherwise copy (some Drive setups block symlinks)."""
    if dst.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(src.resolve(), dst)
    except OSError:
        import shutil
        shutil.copy2(src, dst)
    return True


def build_split_imagefolder_layout(
    images_dir,
    df,
    out_root,
    per_class=200,
    train_frac=0.70,
    val_frac=0.15,
    test_frac=0.15,
    seed=42,
):
    """Create out_root/{train,val,test}/<class>/*.jpg for ImageFolder."""
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-6
    images_dir = Path(images_dir)
    out_root = Path(out_root)
    summary = {}

    for label in sorted(df["label"].unique()):
        rows = df[df["label"] == label].sample(frac=1, random_state=seed)
        if len(rows) > per_class:
            rows = rows.head(per_class)

        n = len(rows)
        n_train = int(train_frac * n)
        n_val = int(val_frac * n)
        n_test = n - n_train - n_val
        splits = {
            "train": rows.iloc[:n_train],
            "val": rows.iloc[n_train : n_train + n_val],
            "test": rows.iloc[n_train + n_val :],
        }

        summary[label] = {}
        for split_name, split_rows in splits.items():
            linked = 0
            for _, row in split_rows.iterrows():
                src = images_dir / f"{int(row.asset_id)}.jpg"
                dst = out_root / split_name / label / f"{int(row.asset_id)}.jpg"
                if src.exists() and _link_image(src, dst):
                    linked += 1
            summary[label][split_name] = linked
    return summary

df = load_labeled_table(
    RAW_ROOT / "gz2_filename_mapping.csv",
    RAW_ROOT / "gz2_hart16.csv",
)
print("Joined rows:", len(df))
print("\nLabel counts:")
print(df["label"].value_counts())
print("\nExample rows:")
print(df[["asset_id", "objid", "gz2_class", "label"]].head())

## Step 4 — Sort into an ImageFolder layout (train / val / test)

`ImageFolder` needs:

```
galaxy_data/
├── train/
│   ├── elliptical/   123.jpg
│   ├── spiral/       456.jpg
│   └── spiral_barred/ ...
├── val/
│   └── (same class subfolders)
└── test/
    └── (same class subfolders)
```

We **symlink** (or copy) files from the flat folder into those paths — we do **not** move the originals. Each class gets `PER_CLASS` galaxies, split 70% / 15% / 15%.

In [ ]:
PER_CLASS = 200  # increase once the pipeline works (e.g. 2000)

summary = build_split_imagefolder_layout(
    IMAGES_DIR,
    df,
    DATA_ROOT,
    per_class=PER_CLASS,
    train_frac=0.70,
    val_frac=0.15,
    test_frac=0.15,
    seed=42,
)
print("Linked images per class and split:")
print(pd.DataFrame(summary).fillna(0).astype(int))

for split in ("train", "val", "test"):
    split_dir = DATA_ROOT / split
    classes = sorted(p.name for p in split_dir.iterdir() if p.is_dir()) if split_dir.exists() else []
    n_imgs = sum(1 for _ in split_dir.rglob("*.jpg")) if split_dir.exists() else 0
    print(f"{split:5s}: {n_imgs:4d} images  classes={classes}")

## Step 5 — Transforms pipeline

In [ ]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

## Step 6 — Three ImageFolder datasets

Point `ImageFolder` at each split folder. Class indices are assigned **alphabetically** and should match across splits if every split contains the same class names.

In [ ]:
train_ds = ImageFolder(root=DATA_ROOT / "train", transform=transform)
val_ds   = ImageFolder(root=DATA_ROOT / "val",   transform=transform)
test_ds  = ImageFolder(root=DATA_ROOT / "test",  transform=transform)

for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    print(f"{name:5s}  n={len(ds):4d}  classes={ds.classes}")

print("class_to_idx:", train_ds.class_to_idx)

## Step 7 — A single sample

In [ ]:
image, label = train_ds[0]
print("shape :", image.shape)
print("dtype :", image.dtype)
print("label :", label, "->", train_ds.classes[label])

## Step 8 — DataLoaders (one per split)

In [ ]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

images, labels = next(iter(train_loader))
print("train batch images:", images.shape)   # (32, 3, 64, 64)
print("train batch labels:", labels.shape)     # (32,)

## Step 9 — Plot a training batch

In [ ]:
images_show = images * 0.5 + 0.5
grid = torchvision.utils.make_grid(images_show[:16], nrow=4)

plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.axis("off")
plt.title("A batch of galaxies (train split)")
plt.show()

print("Labels:", [train_ds.classes[i] for i in labels[:16].tolist()])

## Stretch Goal — Real per-channel mean/std

Compute statistics on the **training split only** (never leak val/test into normalisation).

In [ ]:
stat_ds = ImageFolder(
    root=DATA_ROOT / "train",
    transform=transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()]),
)
stat_loader = DataLoader(stat_ds, batch_size=64, shuffle=False, num_workers=2)

n_pixels = 0
channel_sum = torch.zeros(3)
channel_sq_sum = torch.zeros(3)
for imgs, _ in stat_loader:
    channel_sum += imgs.sum(dim=[0, 2, 3])
    channel_sq_sum += (imgs ** 2).sum(dim=[0, 2, 3])
    n_pixels += imgs.shape[0] * imgs.shape[2] * imgs.shape[3]

mean = channel_sum / n_pixels
std = (channel_sq_sum / n_pixels - mean ** 2).sqrt()
print("per-channel mean:", mean.tolist())
print("per-channel std :", std.tolist())

## Stretch Goal — Augmentation preview

In [ ]:
aug = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(180),
    transforms.ToTensor(),
])

img_path, _ = train_ds.samples[0]
from PIL import Image
pil_img = Image.open(img_path).convert("RGB")

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax in axes:
    ax.imshow(aug(pil_img).permute(1, 2, 0).numpy())
    ax.axis("off")
fig.suptitle("Same galaxy, five random augmentations")
plt.show()

## Reflection (example answers)

1. **Most confusing part.** Raw Kaggle data is flat + CSV. The fix: merge `gz2_filename_mapping.csv` with `gz2_hart16.csv` on `objid`, map `gz2_class` → high-level label, then symlink into `train/val/test/<class>/`.
2. **Recognising a spiral.** Curved arms, central bulge, possibly dust lanes — multi-scale local features.
3. **Hardest pair.** `spiral_barred` vs `spiral` when the bar is faint.
4. **Why the pipeline first.** Wrong joins or splits silently ruin models; fix data before adding layers.

---

That completes Week 1. Next up: **Week 2 — Baselines & Fully-Connected Networks**.